# Story 7: Large Jobs which Each Fan Out into Many Small Tasks
**Who:** A researcher running several sampling campaigns, one per chemical composition. Each campaign consists of
hundreds of short calculations and should run in its own large allocation.

**What they want:** Submit the campaigns from the login node like any other job, and inside each job distribute the
small calculations over the cores of that job - without writing a batch script per campaign.

**Why plain SLURM is painful:** Submitting the small calculations as individual jobs floods the queue. Requesting one
allocation per campaign means writing a batch script which starts and tracks many `srun` steps by hand, and passing
the results of each campaign back to the script on the login node through files.

**How executorlib helps:** The two executors combine. The `SlurmClusterExecutor` on the login node submits one job per
campaign. The submitted function itself creates a `SlurmJobExecutor`, which distributes the small calculations as
`srun` job steps within the allocation of its job.

This notebook runs on the **login node** of a SLURM cluster.

## Cluster-specific settings
**Adapt the following cell to your cluster.** It collects everything which depends on the cluster rather than on your science. The values below are the ones used to test this notebook on a single-node SLURM test cluster. The submission template is a [Jinja2](https://jinja.palletsprojects.com) template which executorlib renders for every function call: add the lines your cluster requires, for example `#SBATCH --partition=<name>`, `#SBATCH --account=<project>` or `module load ...` commands before `{{command}}`. The [executorlib-cluster-setup](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html#setup-with-claude-code) skill for Claude Code can analyse your cluster and generate a tested template.

`slurm_cmd_args` are the additional `srun` arguments used by the `SlurmJobExecutor` inside each job, and `cores_per_campaign` is the size of the allocation requested for each campaign.

In [ ]:
# Cluster-specific settings - adapt these values to your cluster.
submission_template = """\
#!/bin/bash
#SBATCH --output=time.out
#SBATCH --job-name={{job_name}}
#SBATCH --chdir={{working_directory}}
#SBATCH --get-user-env=L
#SBATCH --ntasks={{cores}}
{%- if run_time_max %}
#SBATCH --time={{ [1, run_time_max // 60]|max }}
{%- endif %}
{%- if dependency_list %}
#SBATCH --dependency=afterok:{{ dependency_list | join(',') }}
{%- endif %}

{{command}}
"""
slurm_cmd_args = ["--mpi=pmi2"]  # additional srun arguments for the job steps inside each campaign
cores_per_campaign = 2  # CPU cores requested for each campaign job

## Step 1: The small task and the campaign
`run_campaign()` is executed inside a SLURM job. It uses all cores of its job for the small `sample_energy()` tasks and
returns only the aggregated result, so the cache on the shared file system only stores one result per campaign.

In [ ]:
import time


def sample_energy(composition, seed):
    # Stand-in for a short calculation, e.g. the energy of one random configuration.
    time.sleep(0.5)
    return -3.0 * composition + 0.01 * ((seed * 7919) % 13)


def run_campaign(composition, n_samples, max_cores, slurm_cmd_args):
    # Executed as one SLURM job: distribute the samples over the cores of this job.
    from executorlib import SlurmJobExecutor

    with SlurmJobExecutor(
        max_cores=max_cores,
        resource_dict={"cores": 1, "slurm_cmd_args": slurm_cmd_args},
    ) as exe:
        future_lst = [exe.submit(sample_energy, composition, seed) for seed in range(n_samples)]
        energy_lst = [future.result() for future in future_lst]
    return {"composition": composition, "n_samples": n_samples, "mean_energy": sum(energy_lst) / n_samples}

## Step 2: Request the allocation for each campaign
The campaign function itself is a single serial Python process, so it is submitted with `cores=1` - with more cores
executorlib would start it as an MPI program on every rank. The size of the allocation for the small tasks is
requested with `threads_per_core`: the `SlurmClusterExecutor` requests `cores * threads_per_core` tasks in the
submission template (`#SBATCH --ntasks={{cores}}`), and the `SlurmJobExecutor` inside the job uses them.

In [ ]:
campaign_resources = {"cores": 1, "threads_per_core": cores_per_campaign, "run_time_max": 600}

## Step 3: Submit the campaigns

In [ ]:
from executorlib import SlurmClusterExecutor

cache_directory = "./story7_cache"
composition_lst = [0.1, 0.2]

with SlurmClusterExecutor(
    cache_directory=cache_directory,
    resource_dict={"submission_template": submission_template},
) as exe:
    campaign_lst = [
        exe.submit(
            run_campaign,
            composition,
            n_samples=4,
            max_cores=cores_per_campaign,
            slurm_cmd_args=slurm_cmd_args,
            resource_dict=campaign_resources,
        )
        for composition in composition_lst
    ]
    result_lst = [campaign.result() for campaign in campaign_lst]

result_lst

All features of the `SlurmClusterExecutor` apply to the campaigns: you can disconnect after the submission and
reconnect later ([Story 1](story-1-submit-and-reconnect.ipynb)), and campaigns can depend on each other
([Story 2](story-2-multi-resource-pipeline.ipynb)). `sacct -j <queue_id>` lists the job steps (`<queue_id>.0`,
`<queue_id>.1`, ...) created by the `SlurmJobExecutor` inside each campaign job.

## Thousands of tasks per campaign: nesting flux
Each `srun` job step is registered in the central SLURM database, which becomes a bottleneck for thousands of short
tasks per job. In this case start the hierarchical [flux](https://flux-framework.org) scheduler inside every campaign
job by wrapping the command in the submission template, and use the `FluxJobExecutor` inside `run_campaign()`:

```
#SBATCH --ntasks={{cores}}

srun flux start {{command}}
```

```python
def run_campaign(composition, n_samples, max_cores):
    from executorlib import FluxJobExecutor

    with FluxJobExecutor(max_cores=max_cores) as exe:
        ...
```

flux can be nested further: with `flux_executor_nesting=True` every function submitted to a `FluxJobExecutor` gets
its own flux instance and can again create a `FluxJobExecutor` for its own sub-tasks. See
[SLURM with flux](https://executorlib.readthedocs.io/en/latest/3-hpc-job.html#slurm-with-flux) and [nested executors](https://executorlib.readthedocs.io/en/latest/3-hpc-job.html#nested-executors).

## Clean up

In [ ]:
import shutil

shutil.rmtree(cache_directory, ignore_errors=True)

## What you learned
* A function submitted with the `SlurmClusterExecutor` can create a `SlurmJobExecutor` to use the cores of its own job.
* Serial driver functions are submitted with `cores=1`; the size of their allocation is requested with
  `threads_per_core` (or a custom variable of your submission template).
* For thousands of tasks per job, flux can be started inside each job with `srun flux start`.

**Reference:** [combine both](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html#combine-both), [HPC Job Executor](https://executorlib.readthedocs.io/en/latest/3-hpc-job.html) and
[HPC Cluster Executor](https://executorlib.readthedocs.io/en/latest/2-hpc-cluster.html).